# IFBA Portal Accessibility Analysis

This notebook consumes the result files in `results/` directly (produced by the
collection suite with axe-core and IBM Equal Access Checker) and reproduces the
full Chapter 5 analysis: overall panorama, violations per tool, manual-review
items, distribution by WCAG criterion and POUR principle, impact profile, and
the comparison between the two engines.

All aggregation logic lives in the tested `a11y` package. Here we only load the
data, assemble the tables and figures, and display them. Re-running this
notebook after a new collection updates every number and figure automatically.

To emit the markdown versions of the tables (ready to transcribe into the
thesis text), set `EXPORT = True` in the configuration cell and re-run.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd()
if (cwd / 'analysis').exists():
    sys.path.insert(0, str(cwd / 'analysis'))
    base = cwd
else:
    base = cwd.parent

from a11y import aggregate, convergence
from a11y.load import load_results

RESULTS_DIR = base / 'results' / 'archive'
FIGURES_DIR = base / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

results = load_results(RESULTS_DIR)

DEVICE_LABELS = {'desktop': 'Desktop (1280×720)', 'mobile': 'Mobile (375×667)'}
BUCKET_LABELS = {
    'violation': 'Confirmed violations',
    'potentialviolation': 'Potential violations',
    'recommendation': 'Recommendations',
    'potentialrecommendation': 'Potential recommendations',
    'manual': 'Manual review',
    'pass': 'Passed items',
}

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 110

EXPORT = False


def caption(text):
    from IPython.display import Markdown, display
    display(Markdown(text))


def show(df):
    from IPython.display import display
    display(df)
    if EXPORT:
        print(df.to_markdown(index=False))

## Execution context

Metadata embedded in each result file, making the run self-contained and
auditable.

In [ ]:
context = pd.DataFrame([
    {
        'Device': DEVICE_LABELS[r.metadata.device],
        'Tool': r.metadata.tool,
        'Tool version': r.metadata.tool_version,
        'Browser': f'{r.metadata.browser} {r.metadata.browser_version}',
        'URL': r.metadata.url,
        'Timestamp': r.metadata.timestamp,
    }
    for r in results
])
show(context)

## Overall violation panorama

For axe-core we report distinct rules violated and element-level occurrences.
For IBM, the confirmed violations (items at the *violation* level). The
taxonomies differ, so the numbers should not be compared directly.

In [ ]:
overview = pd.DataFrame(aggregate.overview(results)).rename(columns={
    'device': 'Viewport',
    'axe_rules': 'axe-core (rules)',
    'axe_occurrences': 'axe-core (occurrences)',
    'ibm_violations': 'IBM (violations)',
})
overview['Viewport'] = overview['Viewport'].map(DEVICE_LABELS)
show(overview)

## Violations found by axe-core

Rule, associated WCAG criterion, conformance level, impact assigned by the tool,
and occurrences per *viewport*.

In [ ]:
axe = pd.DataFrame(aggregate.axe_rules(results))
axe['wcag'] = axe['wcag'].str.join(' / ')
axe = axe.rename(columns={
    'rule_id': 'Rule', 'wcag': 'WCAG criterion', 'level': 'Level',
    'impact': 'Impact', 'desktop': 'Desktop', 'mobile': 'Mobile',
})
show(axe)

In [ ]:
axe_fig = pd.DataFrame(aggregate.axe_rules(results)).set_index('rule_id')[['desktop', 'mobile']]
ax = axe_fig.plot.bar(rot=0)
ax.set_xlabel('Rule')
ax.set_ylabel('Occurrences')
ax.set_title('axe-core occurrences by rule and viewport')
ax.legend(['Desktop', 'Mobile'])
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'axe-rules.pdf')
plt.show()

## Violations found by the IBM Equal Access Checker

IBM organizes its findings under its own rule identifiers, and the WCAG
criterion is assigned from the mapping maintained in the analysis package.

In [ ]:
ibm = pd.DataFrame(aggregate.ibm_rules(results))
ibm['wcag'] = ibm['wcag'].str.join(' / ')
ibm = ibm.rename(columns={
    'rule_id': 'ruleId', 'wcag': 'WCAG criterion', 'level': 'Level',
    'desktop': 'Desktop', 'mobile': 'Mobile',
})
show(ibm)

## Items by category (IBM): manual-review context

Beyond the confirmed violations, IBM reports a substantial volume of items that
require human judgement, taken from the `summary.counts` object. They reinforce
that automated evaluation covers only a fraction of the WCAG criteria.

In [ ]:
buckets = pd.DataFrame(aggregate.ibm_buckets(results))
buckets['category'] = buckets['category'].map(BUCKET_LABELS)
buckets = buckets.rename(columns={'category': 'Category', 'desktop': 'Desktop', 'mobile': 'Mobile'})
show(buckets)

In [ ]:
review = pd.DataFrame(aggregate.ibm_buckets(results))
review = review[review['category'] != 'pass'].copy()
review['category'] = review['category'].map(BUCKET_LABELS)
ax = review.set_index('category')[['desktop', 'mobile']].plot.barh()
ax.set_xlabel('Items')
ax.set_ylabel('')
ax.set_title('IBM items by category (excluding passed)')
ax.legend(['Desktop', 'Mobile'])
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ibm-buckets.pdf')
plt.show()

## Distribution by WCAG criterion

Occurrences aggregated by WCAG criterion and the corresponding POUR principle,
for each engine.

In [ ]:
def criterion_table(tool):
    df = pd.DataFrame(aggregate.by_criterion(results, tool))
    return df.rename(columns={
        'criterion': 'WCAG criterion', 'principle': 'Principle',
        'level': 'Level', 'occurrences': 'Occurrences',
    })

caption('**axe-core**')
show(criterion_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(criterion_table('ibm-equal-access'))

### Aggregation by POUR principle

The same occurrences consolidated across the four WCAG principles (Perceivable,
Operable, Understandable, Robust), fulfilling the categorization by principle
set out in the objectives.

In [ ]:
def principle_table(tool):
    df = pd.DataFrame(aggregate.by_principle(results, tool))
    return df.rename(columns={'principle': 'Principle', 'occurrences': 'Occurrences'})

caption('**axe-core**')
show(principle_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(principle_table('ibm-equal-access'))

In [ ]:
prin = pd.DataFrame(aggregate.by_principle(results, 'axe-core'))
ax = prin.plot.bar(x='principle', y='occurrences', rot=0, legend=False)
ax.set_xlabel('POUR principle')
ax.set_ylabel('Occurrences')
ax.set_title('axe-core occurrence distribution by POUR principle')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pour-principles.pdf')
plt.show()

### Aggregation by conformance level

Occurrences consolidated across conformance levels A and AA, the two levels
evaluated in this study.

In [ ]:
def level_table(tool):
    df = pd.DataFrame(aggregate.by_level(results, tool))
    return df.rename(columns={'level': 'Level', 'occurrences': 'Occurrences'})

caption('**axe-core**')
show(level_table('axe-core'))
caption('**IBM Equal Access Checker**')
show(level_table('ibm-equal-access'))

## Impact profile (axe-core)

Distribution of axe-core occurrences by the impact level assigned by the tool.
IBM does not classify findings by severity, so the impact profile is based on
axe-core.

In [ ]:
impact = pd.DataFrame(aggregate.by_impact(results)).rename(
    columns={'impact': 'Impact', 'occurrences': 'Occurrences'})
show(impact)

## Convergences and divergences between the engines

Since axe-core and IBM use distinct rule taxonomies, the comparison is made by
WCAG criterion: criteria flagged by both engines (convergence) and criteria
exclusive to each engine (tool-specific blind spots).

In [ ]:
partition = convergence.compare(results)
comparison = pd.DataFrame([
    {'Group': 'Both engines', 'WCAG criteria': ' / '.join(partition['shared'])},
    {'Group': 'axe-core only', 'WCAG criteria': ' / '.join(partition['axe_only'])},
    {'Group': 'IBM only', 'WCAG criteria': ' / '.join(partition['ibm_only'])},
])
show(comparison)

## Desktop vs. mobile comparison

WCAG criteria flagged in each browsing context, considering both engines
together: criteria common to both *viewports* and criteria specific to each
modality. Barriers exclusive to one *viewport* point to issues sensitive to the
browsing context, such as minimum touch-target size.

In [ ]:
partition = aggregate.viewport_criteria(results)
viewports = pd.DataFrame([
    {'Group': 'Both viewports', 'WCAG criteria': ' / '.join(partition['shared'])},
    {'Group': 'Desktop only', 'WCAG criteria': ' / '.join(partition['desktop_only'])},
    {'Group': 'Mobile only', 'WCAG criteria': ' / '.join(partition['mobile_only'])},
])
show(viewports)

## Prioritization for recommendations

axe-core rules ordered by impact severity and, within each impact level, by
occurrence frequency. This ordering supports proposing improvement
recommendations for the highest-impact and most frequent violations. IBM does
not classify findings by severity, so the prioritization is based on axe-core.

In [ ]:
prio = pd.DataFrame(aggregate.priorities(results))
prio['wcag'] = prio['wcag'].str.join(' / ')
prio = prio.rename(columns={
    'rule_id': 'Rule', 'wcag': 'WCAG criterion', 'level': 'Level',
    'impact': 'Impact', 'occurrences': 'Occurrences',
})
show(prio)

## Summary of findings

Consolidation of the main figures from the previous sections. The values are
computed from the loaded results, so they track new collections automatically.

In [ ]:
ov = {r['device']: r for r in aggregate.overview(results)}
conv = convergence.compare(results)
vp = aggregate.viewport_criteria(results)
principles = [p['principle'] for p in aggregate.by_principle(results, 'axe-core')]
top = aggregate.priorities(results)[0]

summary = f'''
- **Panorama:** axe-core flagged {ov['desktop']['axe_rules']} rule(s) and {ov['desktop']['axe_occurrences']} occurrence(s) on desktop and {ov['mobile']['axe_rules']} rule(s) and {ov['mobile']['axe_occurrences']} occurrence(s) on mobile. IBM confirmed {ov['desktop']['ibm_violations']} and {ov['mobile']['ibm_violations']} violations, respectively.
- **POUR principles affected (axe-core):** {', '.join(principles)}.
- **Engine complementarity:** {len(conv['shared'])} criterion(s) flagged by both, {len(conv['axe_only'])} exclusive to axe-core and {len(conv['ibm_only'])} exclusive to IBM.
- **Browsing context:** {len(vp['desktop_only'])} criterion(s) exclusive to desktop and {len(vp['mobile_only'])} to mobile.
- **Correction priority:** the highest-impact, most-frequent rule is `{top['rule_id']}` ({top['impact']}, {top['occurrences']} occurrences).
'''
caption(summary)